In [1]:
# Cell 1: Import libraries and load data
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load Titanic dataset directly from URL
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# First look at the data
print(df.shape)
df.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
# Cell 2: Check missing values
print(df.isnull().sum())
print("\nData types:\n", df.dtypes)

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Data types:
 PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object


In [3]:
# Cell 3: Handle missing data
df_clean = df.copy()

# Drop Cabin (too many missing values to be useful)
df_clean = df_clean.drop(columns=['Cabin'])

# Drop columns that don't help prediction
df_clean = df_clean.drop(columns=['PassengerId', 'Name', 'Ticket'])

# Fill missing Age with median (robust to outliers)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())

# Fill missing Embarked with mode (most frequent value)
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

# Confirm no missing values remain
print(df_clean.isnull().sum())
print("\nShape after cleaning:", df_clean.shape)

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

Shape after cleaning: (891, 8)


In [4]:
# Cell 4: Encode categorical variables
# Sex: binary -> use Label Encoding (male/female -> 0/1)
le = LabelEncoder()
df_clean['Sex'] = le.fit_transform(df_clean['Sex'])

# Embarked: 3 categories (S, C, Q) -> use One-Hot Encoding
df_clean = pd.get_dummies(df_clean, columns=['Embarked'], drop_first=True)

print(df_clean.head())
print("\nColumns now:", df_clean.columns.tolist())

   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked_Q  Embarked_S
0         0       3    1  22.0      1      0   7.2500       False        True
1         1       1    0  38.0      1      0  71.2833       False       False
2         1       3    0  26.0      0      0   7.9250       False        True
3         1       1    0  35.0      1      0  53.1000       False        True
4         0       3    1  35.0      0      0   8.0500       False        True

Columns now: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_Q', 'Embarked_S']


In [5]:
# Cell 5: Split into train/test sets
X = df_clean.drop(columns=['Survived'])  # features
y = df_clean['Survived']                  # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Training set shape: (712, 8)
Testing set shape: (179, 8)


In [6]:
# Cell 6: Standardize numerical features
# Important: fit the scaler ONLY on training data, then apply to both
# This prevents "data leakage" from the test set into training

numeric_cols = ['Age', 'Fare']

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(X_train.head())

     Pclass  Sex       Age  SibSp  Parch      Fare  Embarked_Q  Embarked_S
331       1    1  1.253641      0      0 -0.078684       False        True
733       2    1 -0.477284      0      0 -0.377145       False        True
382       3    1  0.215086      0      0 -0.474867       False        True
704       3    1 -0.246494      1      0 -0.476230       False        True
813       3    0 -1.785093      4      2 -0.025249       False        True


In [7]:
# Cell 7: Summary
print("="*50)
print("TASK 1 SUMMARY: Data Preprocessing")
print("="*50)
print(f"Original dataset shape: {df.shape}")
print(f"Final cleaned dataset shape: {df_clean.shape}")
print(f"Missing values handled: Age (median), Embarked (mode), Cabin (dropped)")
print(f"Categorical encoding: Sex (Label), Embarked (One-Hot)")
print(f"Numerical features scaled: Age, Fare")
print(f"Train set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")

TASK 1 SUMMARY: Data Preprocessing
Original dataset shape: (891, 12)
Final cleaned dataset shape: (891, 9)
Missing values handled: Age (median), Embarked (mode), Cabin (dropped)
Categorical encoding: Sex (Label), Embarked (One-Hot)
Numerical features scaled: Age, Fare
Train set: 712 samples | Test set: 179 samples
